# Region Group Analysis - Open Source Alternative to ArcGIS Pro

This notebook demonstrates how to use the `region_group.py` script to identify connected regions in binary rasters.

**Replaces ArcGIS Pro workflow:**
1. ~~Upload raster to ArcGIS Pro~~
2. ~~Region Group (Spatial Analyst Tools)~~
3. ~~Raster to Polygon~~
4. ~~Export features~~

**New Python workflow:**
1. Load binary raster
2. Run `RegionGroup.region_group()`
3. Convert to polygons
4. Export as GeoPackage/Shapefile

---

**Author:** Nalaquq LLC / QCORP GIS Training  
**Date:** December 2025

## Setup

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our region grouping module
from region_group import RegionGroup, region_group_workflow

# Configure visualization
plt.rcParams['figure.figsize'] = (14, 10)
%matplotlib inline

print("✓ Setup complete")

## Example 1: Basic Region Grouping (Replaces ArcGIS Region Group)

This example shows how to identify all connected water bodies in a binary water/land raster.

In [ ]:
# Path to your binary water/land raster from Google Earth Engine
# 0 = water, 1 = land
raster_path = "path/to/WaterLand_Classification.tif"

# Initialize RegionGroup
rg = RegionGroup(raster_path)

### Run Region Grouping

This is equivalent to ArcGIS Pro's Region Group tool with:
- **Number of neighbors:** 8
- **Zone grouping:** WITHIN (group connected pixels of same value)
- **Excluded value:** 1 (land) - only group water pixels

In [ ]:
# Run region grouping (equivalent to ArcGIS Region Group)
labeled = rg.region_group(
    neighbors=8,           # 8-neighbor connectivity (includes diagonals)
    zone_grouping='within', # Group pixels with same value
    excluded_value=1       # Exclude land pixels (only group water)
)

print(f"\nIdentified {rg.num_regions} connected water bodies")

### Calculate Region Statistics

In [ ]:
# Get statistics for each region
stats = rg.calculate_region_statistics()

# Display top 10 largest regions
print("\nTop 10 Largest Water Bodies:")
print(stats[['region_id', 'area_ha', 'pixel_count']].head(10))

# Show histogram of region sizes
fig, ax = plt.subplots(figsize=(10, 6))
stats['area_ha'].hist(bins=50, ax=ax, edgecolor='black')
ax.set_xlabel('Region Area (hectares)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Water Body Sizes', fontsize=14, fontweight='bold')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

### Visualize Regions

In [ ]:
# Visualize the labeled regions
rg.visualize_regions(show_labels=True, figsize=(16, 8))

## Example 2: Extract Specific River System

Extract the largest water body (main river) and filter out small features.

In [ ]:
# Get the 3 largest water bodies
largest_ids = rg.get_largest_regions(n=3)
print(f"Largest region IDs: {largest_ids}")

# Get mask for largest region only
main_river_mask = rg.get_region_by_id(largest_ids[0])

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.imshow(rg.labeled_regions > 0, cmap='Blues')
ax1.set_title('All Water Bodies', fontsize=14, fontweight='bold')
ax1.axis('off')

ax2.imshow(main_river_mask, cmap='Blues')
ax2.set_title('Main River (Largest Region)', fontsize=14, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Example 3: Filter Regions by Size

Remove small isolated water bodies and focus on significant river channels.

In [ ]:
# Filter regions: Keep only water bodies > 0.5 hectares (5,000 m²)
filtered_mask = rg.filter_regions(
    min_area_m2=5000,
    value=0  # Only water regions
)

# Compare original vs filtered
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.imshow(rg.labeled_regions > 0, cmap='Blues')
ax1.set_title(f'Original ({rg.num_regions} regions)', fontsize=14, fontweight='bold')
ax1.axis('off')

ax2.imshow(filtered_mask, cmap='Blues')
filtered_count = len(np.unique(rg.labeled_regions[filtered_mask])) - 1
ax2.set_title(f'Filtered (> 0.5 ha, {filtered_count} regions)', fontsize=14, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Example 4: Convert Regions to Polygons

Convert labeled regions to vector polygons (equivalent to Raster to Polygon tool).

In [ ]:
# Convert all regions to polygons
gdf = rg.regions_to_polygons(
    region_ids=None,  # None = all regions
    simplify_tolerance=1.0  # Simplify to 1 meter
)

print(f"\nCreated {len(gdf)} polygon features")
print(gdf.head())

# Plot polygons
fig, ax = plt.subplots(figsize=(12, 10))
gdf.plot(ax=ax, facecolor='blue', edgecolor='darkblue', alpha=0.6, linewidth=1.5)
ax.set_title('Water Body Polygons', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Example 5: Export Results

Save the labeled regions and polygons for use in ArcGIS Pro or QGIS.

In [ ]:
# Create output directory
output_dir = Path("region_group_output")
output_dir.mkdir(exist_ok=True)

# 1. Export labeled raster
labeled_raster_path = rg.export_labeled_raster(
    output_dir / "water_regions_labeled.tif",
    dtype='int32'
)

# 2. Export statistics
stats_path = rg.export_statistics(
    output_dir / "water_regions_stats.csv",
    format='csv'
)

# 3. Export polygons as GeoPackage
polygon_path = output_dir / "water_regions.gpkg"
gdf.to_file(polygon_path, driver='GPKG')

print("\n✓ Exports complete!")
print(f"\nGenerated files:")
print(f"  - {labeled_raster_path}")
print(f"  - {stats_path}")
print(f"  - {polygon_path}")
print("\nYou can now open these in ArcGIS Pro or QGIS!")

## Example 6: Complete Workflow (One Function)

Use the convenience function to run the entire workflow.

In [ ]:
# Run complete workflow with one function
results = region_group_workflow(
    raster_path="path/to/WaterLand_Classification.tif",
    output_dir="region_group_output",
    neighbors=8,              # 8-neighbor connectivity
    excluded_value=1,         # Exclude land (value=1)
    min_area_m2=1000,        # Filter regions < 0.1 hectare
    export_polygons=True,
    export_labeled_raster=True,
    visualize=True           # Show plots
)

print("\n✓ Complete workflow finished!")

## Example 7: Integration with Existing River Extraction

Combine region grouping with the existing `river_extraction.py` workflow.

In [ ]:
# Import existing river extraction module
from river_extraction import RiverExtractor

# Step 1: Region grouping to identify all water bodies
rg = RegionGroup("path/to/WaterLand_Classification.tif")
labeled = rg.region_group(neighbors=8, excluded_value=1)

# Step 2: Get largest region (main river)
largest_id = rg.get_largest_regions(n=1)[0]
main_river_mask = rg.get_region_by_id(largest_id)

# Step 3: Use RiverExtractor to convert to clean polygons
extractor = RiverExtractor("path/to/WaterLand_Classification.tif")
river_gdf = extractor.extract_river(
    method='largest',
    min_size_pixels=50,
    morphology_iterations=1,
    simplify_tolerance=1.0
)

# Save final river polygon
extractor.save_vector(river_gdf, "main_river_extracted.gpkg")

print("\n✓ Integrated workflow complete!")

## Comparison: ArcGIS Pro vs Open Source

| Step | ArcGIS Pro | Open Source Python |
|------|-----------|--------------------|
| 1. Load raster | Manual import | `RegionGroup(raster_path)` |
| 2. Region grouping | Region Group tool (GUI) | `rg.region_group(neighbors=8, excluded_value=1)` |
| 3. Statistics | Zonal Statistics tool | `rg.calculate_region_statistics()` |
| 4. Raster to Polygon | Raster to Polygon tool | `rg.regions_to_polygons()` |
| 5. Export | Manual export | `rg.export_labeled_raster()`, `gdf.to_file()` |
| **Total clicks** | ~20+ | **3 lines of code** |
| **Cost** | ArcGIS Pro license ($700/yr) | **Free (open source)** |
| **Automation** | Model Builder required | Native Python scripting |

---

## Next Steps

1. **Batch processing**: Process multiple dates for temporal analysis
2. **Threshold testing**: Test different NIR thresholds in GEE
3. **Validation**: Compare results with ArcGIS Pro output
4. **Integration**: Add to your temporal analysis workflow

## Questions?

Contact the QCORP GIS Training team or open an issue in the GitHub repository.